# 09 · 例外與檔案 I/O

本書教你在程式出錯時優雅地處理例外，並用正確的方式讀寫各種格式的檔案與管理路徑。

## 學習目標
- 理解例外處理（exception handling）的流程，能用 try/except 攔截並回應錯誤而不讓程式中斷。
- 能使用 with 上下文管理器（context manager）安全地管理檔案等資源，確保正確釋放。
- 掌握純文字檔的讀寫，並理解編碼（encoding）對中文等文字的重要性。
- 能用物件式路徑（pathlib.Path）與傳統 os/os.path 操作檔案系統，並用 glob 搜尋檔案。
- 能在 JSON 與 CSV 這兩種常見資料格式之間進行序列化（serialization）與還原。
- 認識 warnings 警告控制與其他序列化工具（yaml、pickle、base64）的適用情境。

## 例外處理基礎

例外（exception）是程式執行中發生的錯誤事件，例如除以零或型別不符。預設情況下未處理的例外會讓程式整個中斷。

程式出錯是常態，學會攔截特定例外並給予合理回應，比讓程式整個崩潰更穩健。

形狀：
```
try:
    可能出錯的程式
except 某種例外類別 as e:
    出錯時的處理
else:
    沒出錯才執行
finally:
    無論如何都執行
```

常見內建例外對照：

| 例外類別 | 觸發時機 |
|---|---|
| ZeroDivisionError | 除以零 |
| TypeError | 型別不符的運算 |
| ValueError | 型別對但值不合法 |
| Exception | 幾乎所有例外的基底類別 |

In [ ]:
# 概念：用 try/except 攔截特定例外，回傳友善訊息而非讓程式崩潰
def safe_divide(a, b):
    try:
        result = a / b                      # 可能引發 ZeroDivisionError 或 TypeError
    except ZeroDivisionError:               # 多重 except：先攔最具體的例外
        return "錯誤：除數不能為零"
    except TypeError:                       # 輸入不是數字時觸發
        return "錯誤：輸入必須是數字"
    else:
        return f"結果是 {result}"          # 沒出錯才執行 else
    finally:
        # 注意：finally 無論成功或失敗都會執行，常用來收尾
        print("  （已完成一次除法嘗試）")

print(safe_divide(10, 2))
print(safe_divide(10, 0))
print(safe_divide(10, "x"))

# 概念：用 raise 主動拋出例外，表達「這個輸入我不接受」
def check_height(h):
    if h <= 0:
        raise ValueError("樓高必須為正數")   # 主動拋出，交給呼叫端處理
    return h

try:
    check_height(-5)
except ValueError as e:                      # as e 取得例外物件，可讀訊息
    print("攔截到：", e)

## 上下文管理器與 warnings

上下文管理器（context manager）是搭配 with 使用的物件，能在進入區塊時取得資源、離開時自動釋放，即使中途發生例外也會釋放。

with 能保證資源（如檔案）被正確關閉；warnings 用於提示而非中斷，與例外做區隔。

形狀：
```
with 上下文管理器 as 變數:
    使用資源
# 離開區塊自動清理
```

In [ ]:
# 概念：用 contextlib 自製計時上下文管理器，示範進入與離開區塊的行為
import time
from contextlib import contextmanager

@contextmanager                              # 把產生器函式變成上下文管理器
def timer(label):
    start = time.perf_counter()              # 進入區塊：取得起始時間
    print(f"[{label}] 開始")
    try:
        yield                                # yield 之前是進入、之後是離開
    finally:
        # 技巧：放在 finally 確保區塊內出錯也會印出耗時
        elapsed = time.perf_counter() - start
        print(f"[{label}] 結束，耗時 {elapsed:.4f} 秒")

with timer("累加迴圈"):
    total = sum(i for i in range(100000))
    print("  總和 =", total)

# 概念：用 warnings 對棄用功能發出提示，不中斷程式
import warnings

def old_area(side):
    # 注意：warnings.warn 只是提示，程式會繼續往下跑
    warnings.warn("old_area 已棄用，請改用 square_area", DeprecationWarning)
    return side * side

with warnings.catch_warnings():
    warnings.simplefilter("always")          # 讓 DeprecationWarning 每次都顯示
    print("面積 =", old_area(4))

## 文字檔讀寫與編碼

用 open() 開啟檔案，透過模式（mode）決定讀或寫，再用 read/write 操作內容。

讀寫檔最常見的坑就是編碼（encoding），明確指定 utf-8 可避免中文亂碼；搭配 with 寫出安全的讀寫樣板。

常用模式對照：

| 模式 | 意義 |
|---|---|
| r | 讀取（檔案須存在） |
| w | 寫入（覆蓋既有內容） |
| a | 附加（接在結尾） |

In [ ]:
# 概念：用 with + open() 指定 utf-8 編碼寫入中文，再讀回
# 造一段內建的多行中文字串當示範用
text = "第一行：台北101\n第二行：高雄港\n第三行：日月潭\n"

filename = "demo_text.txt"
with open(filename, "w", encoding="utf-8") as f:   # 注意：明確指定 utf-8 才能正確寫中文
    f.write(text)                                   # w 模式會覆蓋既有內容

# 概念：逐行迭代讀檔，檔案物件本身就可當迭代器
with open(filename, "r", encoding="utf-8") as f:
    for i, line in enumerate(f, start=1):           # 一次拿一行，省記憶體
        print(f"讀到第 {i} 行：{line.rstrip()}")    # rstrip 去掉行尾換行符

# 技巧：一次讀全部用 read()，讀成 list（每元素一行）用 readlines()
with open(filename, "r", encoding="utf-8") as f:
    print("總字元數 =", len(f.read()))

## 路徑管理：pathlib 與 os

物件式路徑（pathlib.Path）把路徑當成物件，用運算子 `/` 組合子路徑，並提供 exists、mkdir 等方法。

跨平台時手動拼接路徑容易出錯，pathlib 提供更直覺安全的寫法，並可對照傳統 os.path 作法。

形狀：
```
Path("資料夾") / "子資料夾" / "檔名.txt"
```

In [ ]:
# 概念：用 pathlib.Path 建立暫存資料夾並組出子檔案路徑
from pathlib import Path

base = Path("demo_workspace")                # 把路徑當物件
base.mkdir(exist_ok=True)                     # 注意：exist_ok=True 避免資料夾已存在時報錯

data_file = base / "buildings.txt"           # 用 / 運算子組路徑，跨平台安全
data_file.write_text("示範內容\n", encoding="utf-8")   # Path 直接寫檔，省去 open

print("資料夾存在嗎？", base.exists())
print("檔案絕對路徑：", data_file.resolve())
print("副檔名：", data_file.suffix, "｜檔名：", data_file.name)

# 概念：對照傳統 os / os.path 的寫法
import os

old_path = os.path.join("demo_workspace", "buildings.txt")   # os.path.join 手動拼接
print("os.path 組出的路徑：", old_path)
print("os.path.exists：", os.path.exists(old_path))

## 檔案搜尋 glob

glob 用萬用字元（wildcard）樣式比對路徑，一次取得符合條件的檔案清單。

實務上常需要一次處理一批檔案，glob 讓你用樣式比對取得符合條件的路徑。

萬用字元對照：

| 樣式 | 意義 |
|---|---|
| `*` | 比對同一層任意字元 |
| `**` | 遞迴比對所有子層 |

In [ ]:
# 概念：先造出幾個不同副檔名的檔案，再用萬用字元樣式篩出特定類型
from pathlib import Path

folder = Path("demo_glob")
folder.mkdir(exist_ok=True)

# 造一批仿真檔案（兩個 csv、兩個 txt）當示範用
for name in ["a.csv", "b.csv", "note.txt", "readme.txt"]:
    (folder / name).write_text("x", encoding="utf-8")

# glob：只比對同一層；用 * 當萬用字元
csv_files = list(folder.glob("*.csv"))       # 篩出所有 .csv
print("找到的 CSV：", [p.name for p in csv_files])

# 技巧：rglob 會遞迴搜尋所有子資料夾，等同 glob('**/...')
all_txt = list(folder.rglob("*.txt"))
print("遞迴找到的 TXT：", [p.name for p in all_txt])

## JSON 序列化

序列化（serialization）是把記憶體中的物件轉成可儲存或傳輸的文字 / 位元組；JSON 是其中最通用的文字格式。

JSON 是設定檔與資料交換的通用格式，學會在字典清單與 JSON 文字之間雙向轉換。

Python 與 JSON 型別對應：

| Python | JSON |
|---|---|
| dict | object |
| list | array |
| str | string |
| True/False/None | true/false/null |

In [ ]:
# 概念：把巢狀字典寫成 JSON 檔再讀回，比較是否一致
import json

# 造一筆內建的巢狀資料當示範用
city = {
    "name": "範例市",
    "buildings": [
        {"name": "市政大樓", "floors": 12},
        {"name": "圖書館", "floors": 5},
    ],
}

with open("city.json", "w", encoding="utf-8") as f:
    # 注意：ensure_ascii=False 中文才不會被轉成 \uXXXX；indent 讓檔案易讀
    json.dump(city, f, ensure_ascii=False, indent=2)

with open("city.json", "r", encoding="utf-8") as f:
    loaded = json.load(f)                    # 從檔案讀回成 Python 物件

print("還原後與原物件相同？", loaded == city)
# 技巧：dumps（多一個 s）回傳字串而非寫檔，方便除錯時直接看內容
print(json.dumps(loaded, ensure_ascii=False))

## CSV 表格資料讀寫

CSV（逗號分隔值）是以列為單位、欄位用分隔符號（delimiter）隔開的純文字表格格式。

CSV 是試算表與表格資料的常見格式，了解用列為單位與用欄位名稱兩種讀寫方式。

兩種讀寫器對照：

| 工具 | 每列形式 |
|---|---|
| csv.reader / writer | 用 list（依位置） |
| csv.DictReader / DictWriter | 用 dict（依欄名） |

In [ ]:
# 概念：用 DictWriter 以欄位名稱寫出 CSV，再用 DictReader 讀回
import csv

# 造幾筆內建人員資料當示範用
people = [
    {"name": "小明", "age": 30, "city": "台北"},
    {"name": "小華", "age": 25, "city": "台中"},
    {"name": "小美", "age": 28, "city": "高雄"},
]
fields = ["name", "age", "city"]             # 表頭 header，決定欄位順序

# 注意：newline='' 是 csv 模組的標準寫法，避免多出空白列
with open("people.csv", "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fields)
    writer.writeheader()                     # 先寫表頭
    writer.writerows(people)                 # 一次寫多列

with open("people.csv", "r", encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f)               # 每列讀成 dict，鍵就是欄名
    for row in reader:
        # 注意：CSV 讀回的值都是字串，數值要自己 int/float 轉換
        print(f"{row['name']} 在 {row['city']}，明年 {int(row['age']) + 1} 歲")

## 其他序列化格式概覽

除了 JSON 與 CSV，還有幾種常見的序列化工具，各有適用情境與風險。建立選型概念，知道何時該用哪一種。

格式選型對照：

| 格式 | 適用情境 | 風險 / 限制 |
|---|---|---|
| YAML | 人類可讀的設定檔 | 需第三方套件、縮排敏感 |
| pickle | Python 物件二進位序列化 | 僅限信任來源，可執行任意程式碼 |
| base64 | 把二進位塞進文字通道 | 不是加密，只是編碼 |

In [ ]:
# 概念：把同一份資料概念性地對應到 pickle 與 base64，著重何時該用哪一種
import pickle
import base64

data = {"floors": 10, "tags": ["住宅", "商辦"]}   # 造一份內建資料當示範用

# pickle：序列化成 bytes，能保留 Python 物件結構
# 注意：pickle 反序列化會執行內容，只可載入信任來源的資料
blob = pickle.dumps(data)
print("pickle 還原：", pickle.loads(blob))

# base64：把二進位 bytes 編成純文字，方便塞進 JSON 或網址
# 注意：base64 只是編碼不是加密，不提供任何保密性
encoded = base64.b64encode(blob).decode("ascii")
print("base64 文字（前 40 字）：", encoded[:40], "...")

# 技巧：YAML 需 pip install pyyaml；此處示意其風格，設定檔常用它因為最易讀
yaml_like = "floors: 10\ntags:\n  - 住宅\n  - 商辦"
print("YAML 風格示意：\n" + yaml_like)

## 練習

以下三題由淺到深，皆為複合型但簡短可快速完成。資料一律自己用 list / numpy 造，不讀任何外部檔案。只需在 TODO 處完成。

In [ ]:
# TODO 1 ·（簡單）建築清單存成 JSON（整合：文字檔讀寫與編碼 + JSON 序列化）
#   情境：把一批建築的樓高與樓層數仿真資料存成設定檔，方便之後讀回。
#   要求：
#     1. 在程式內用 list 自己造出 5 筆建築字典，每筆含中文名稱、樓高 height（公尺）、樓層數 floors（整數）。
#     2. 用 with 搭配 open() 並指定 utf-8 編碼，以 json.dump 寫入檔案，設定 ensure_ascii=False 讓中文正常顯示。
#     3. 再用 with 與 json.load 把檔案讀回成 Python 物件。
#   驗收：讀回後的清單與原始 5 筆資料完全一致，且檔案內的中文不是亂碼。
# TODO: 在這裡完成


In [ ]:
# TODO 2 ·（中間）多份街廓 CSV 彙整（整合：glob + CSV 讀寫 + pathlib + 例外處理）
#   情境：每個街廓 block 一份 CSV，記錄該街廓各建築的樓地板面積 GFA，要彙整成每街廓總量。
#   要求：
#     1. 用 pathlib.Path 建立暫存資料夾，並用內建仿真資料以 csv.DictWriter 寫出 3 份 CSV（表頭：建築 id、GFA）。
#     2. 用 pathlib 的 glob 以萬用字元樣式 *.csv 取得所有檔案清單。
#     3. 逐檔用 csv.DictReader 讀入，累加每份檔案的 GFA 總和；對讀取或數值轉換失敗者用 try/except 攔截並略過。
#     4. 把每街廓的彙整結果整理成字典。
#   驗收：每個街廓檔名對應一個 GFA 總和；故意放入一筆壞資料時程式不中斷而是被略過。
# TODO: 在這裡完成


In [ ]:
# TODO 3 ·（稍難）容積率政策情境前後比較（整合：JSON + 例外處理 + with 上下文管理器 + 條件聚合）
#   情境：建築資料含占地面積 footprint 與樓地板面積 GFA，計算容積率 FAR = GFA / footprint，
#         比較套用高度乘數政策前後，有多少棟超出容積上限。
#   要求：
#     1. 用 numpy 造出 8 棟建築的 footprint 與 floors 仿真陣列，並依樓層推算 GFA。
#     2. 寫一個函式計算每棟 FAR，對 footprint 為 0 用 try/except 攔截除零並標記為無效，不讓程式崩潰。
#     3. 設定容積上限門檻，先統計政策前超標棟數；再對所有 GFA 乘上 1.2 高度乘數模擬放寬，重新統計超標棟數。
#     4. 以 with 開檔並用 json.dump（ensure_ascii=False）把「政策前超標數、政策後超標數、各棟 FAR」寫成 JSON 報告。
#   驗收：政策後超標棟數不少於政策前；JSON 報告能被 json.load 正確讀回、含無效棟位的標記。
# TODO: 在這裡完成


## 小結

- 例外處理用 try/except/else/finally 攔截特定錯誤，並可用 raise 主動拋出，讓程式在出錯時仍可控。
- with 上下文管理器保證資源（尤其是檔案）即使中途出錯也會釋放；warnings 用於提示而非中斷。
- 文字檔讀寫務必明確指定 encoding="utf-8"，是避免中文亂碼最關鍵的一步。
- pathlib.Path 以物件式運算子組路徑，比手動拼接更安全跨平台，並可對照傳統 os.path。
- glob 用萬用字元一次取得整批檔案；JSON 與 CSV 分別是資料交換與表格資料的通用序列化格式。
- 其他格式各有定位：YAML 重可讀、pickle 限信任來源、base64 用於把二進位塞進文字通道。